# [KUNCI FASILITATOR] Chatbot Cerdas dengan AINotebook ini adalah versi lengkap untuk fasilitator. Semua bagian TUGAS sudah terisi.Simpan berkas ini **di luar** folder yang dibagikan ke peserta.Pakai notebook ini untuk:1. Uji coba penuh sebelum hari pelaksanaan2. Demo di depan kelas saat menjelaskan konsep3. Rujukan cepat kalau ada peserta yang tersendat

---## Bagian 0. PersiapanSel di bawah menyiapkan model AI. Kamu tidak perlu mengubah apa pun di sini, cukup jalankan.Pada percobaan pertama, komputer akan mengunduh model berukuran beberapa ratus megabita.Unduhan ini terjadi di server Google, bukan di laptopmu, jadi tidak membebani WiFi ruangan.Tunggu sampai muncul tulisan **MODEL SIAP DIPAKAI**.

In [ ]:
# @title Sel persiapan (jalankan, tidak perlu diubah) { display-mode: "form" }import importlibimport subprocessimport sysNAMA_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"def pastikan_terpasang(nama_paket):    """Memasang pustaka hanya kalau memang belum tersedia."""    try:        importlib.import_module(nama_paket)    except ImportError:        print(f"Memasang {nama_paket} sebentar...")        subprocess.run(            [sys.executable, "-m", "pip", "install", "-q", nama_paket], check=True        )pastikan_terpasang("transformers")# Dua jalur teknis di bawah menghasilkan hasil yang sama persis.# Jalur utama memakai pustaka sentence-transformers kalau tersedia.# Jalur cadangan memakai pustaka transformers yang sudah ada di Colab.try:    from sentence_transformers import SentenceTransformer, util    _model = SentenceTransformer(NAMA_MODEL)    UKURAN_VEKTOR = _model.get_sentence_embedding_dimension()    def ubah_ke_vektor(teks):        """Mengubah satu kalimat atau sekumpulan kalimat menjadi vektor angka."""        return _model.encode(teks, convert_to_tensor=True)    def kemiripan(vektor_a, vektor_b):        """Menghitung kemiripan makna, nilainya antara -1 sampai 1."""        return util.cos_sim(vektor_a, vektor_b)    JALUR = "sentence-transformers"except Exception as galat:    import torch    from transformers import AutoModel, AutoTokenizer    print(f"Memakai jalur cadangan. Penyebab: {type(galat).__name__}")    _tokenizer = AutoTokenizer.from_pretrained(NAMA_MODEL)    _jaringan = AutoModel.from_pretrained(NAMA_MODEL)    _jaringan.eval()    UKURAN_VEKTOR = _jaringan.config.hidden_size    def ubah_ke_vektor(teks):        """Mengubah satu kalimat atau sekumpulan kalimat menjadi vektor angka."""        satu_kalimat = isinstance(teks, str)        daftar = [teks] if satu_kalimat else list(teks)        masukan = _tokenizer(            daftar, padding=True, truncation=True, max_length=128, return_tensors="pt"        )        with torch.no_grad():            keluaran = _jaringan(**masukan)        topeng = masukan["attention_mask"].unsqueeze(-1).float()        rata_rata = (keluaran.last_hidden_state * topeng).sum(1) / topeng.sum(1)        return rata_rata[0] if satu_kalimat else rata_rata    def kemiripan(vektor_a, vektor_b):        """Menghitung kemiripan makna, nilainya antara -1 sampai 1."""        if vektor_a.dim() == 1:            vektor_a = vektor_a.unsqueeze(0)        if vektor_b.dim() == 1:            vektor_b = vektor_b.unsqueeze(0)        vektor_a = torch.nn.functional.normalize(vektor_a, p=2, dim=1)        vektor_b = torch.nn.functional.normalize(vektor_b, p=2, dim=1)        return vektor_a @ vektor_b.T    JALUR = "transformers"import pandas as pdprint()print("=" * 50)print("MODEL SIAP DIPAKAI")print("=" * 50)print("Jalur teknis     :", JALUR)print("Panjang vektor   :", UKURAN_VEKTOR, "angka per kalimat")

---## Bagian 1. Kalimat berubah menjadi angkaKomputer tidak mengerti huruf. Model AI mengubah tiap kalimat menjadi deretan angkayang disebut **vektor**. Kalimat yang maknanya mirip akan punya deretan angka yang mirip juga.Bayangkan sebuah peta. Kalimat dengan makna serupa diletakkan berdekatan di peta itu,kalimat dengan makna berbeda diletakkan berjauhan.Jalankan sel di bawah dan perhatikan hasilnya.

In [ ]:
kalimat_contoh = [    "Di mana letak Candi Borobudur?",    "Borobudur itu ada di daerah mana?",    "Berapa harga tiket bus ke Bandung?",]vektor_contoh = ubah_ke_vektor(kalimat_contoh)print("Jumlah kalimat        :", len(kalimat_contoh))print("Panjang vektor         :", UKURAN_VEKTOR, "angka")print()print("Sepuluh angka pertama dari kalimat nomor 1:")print([round(float(a), 3) for a in vektor_contoh[0][:10]])

In [ ]:
nilai = kemiripan(vektor_contoh, vektor_contoh)print("Skor kemiripan makna, 1.00 berarti sama persis, 0.00 berarti tidak berhubungan")print("-" * 72)for i in range(len(kalimat_contoh)):    for j in range(i + 1, len(kalimat_contoh)):        print(f"skor {float(nilai[i][j]):.3f}")        print(f"   A: {kalimat_contoh[i]}")        print(f"   B: {kalimat_contoh[j]}")        print()

### PerhatikanKalimat 1 dan kalimat 2 memakai susunan kata yang berbeda, tapi skor kemiripannya tinggikarena maksudnya sama. Kalimat 3 membahas hal lain, jadi skornya rendah.Kemampuan inilah yang membuat chatbot kita nanti tetap paham meskipun pertanyaandiketik dengan kalimat yang berbeda dari daftar yang kita siapkan.

---## Bagian 2. Pilih tema chatbotmuSekarang pilih mau membuat chatbot tentang apa. Klik menu di sel bawah, pilih salah satu:| Pilihan | Isi ||---|---|| `wisata` | Tempat wisata terkenal dunia dan Indonesia, seperti Borobundur, Menara Eiffel, Machu Picchu || `sejarah` | Peristiwa sejarah dunia, seperti Perang Dunia, Revolusi Prancis, pendaratan di Bulan |Setelah memilih, jalankan selnya.

In [ ]:
TEMA = "wisata"  # @param ["wisata", "sejarah"]# Sumber data. Ubah tiga baris ini hanya kalau nama akun, repo, atau cabangnya berbeda.GITHUB_AKUN = "FeliksMakarios"GITHUB_REPO = "workshop-chatbot-semantik"GITHUB_CABANG = "main"alamat = (    "https://raw.githubusercontent.com/"    + GITHUB_AKUN + "/" + GITHUB_REPO + "/" + GITHUB_CABANG    + "/data/faq_" + TEMA + ".csv")try:    df = pd.read_csv(alamat)    print("Data berhasil dimuat dari GitHub.")except Exception as galat:    print("Gagal memuat data:", type(galat).__name__)    print("Pakai sel cadangan di Lampiran A di bagian paling bawah notebook ini.")    raiseprint("Tema terpilih          :", TEMA)print("Jumlah pengetahuan awal:", len(df), "pasang tanya jawab")df.head()

In [ ]:
# Melihat tiga contoh isi pengetahuan chatbotfor i in range(3):    print("Tanya :", df["pertanyaan"][i])    print("Jawab :", df["jawaban"][i])    print("-" * 72)

---## Bagian 3. Membangun chatbotChatbot kita bekerja dalam tiga langkah:1. Semua pertanyaan di tabel pengetahuan diubah menjadi vektor, dikerjakan sekali saja di awal2. Pertanyaan yang kamu ketik juga diubah menjadi vektor3. Komputer mencari pertanyaan di tabel yang vektornya paling dekat, lalu mengambil jawabannyaJalankan dua sel di bawah.

In [ ]:
daftar_pertanyaan = df["pertanyaan"].tolist()daftar_jawaban = df["jawaban"].tolist()vektor_pengetahuan = ubah_ke_vektor(daftar_pertanyaan)print(len(daftar_pertanyaan), "pertanyaan sudah diubah menjadi vektor.")

In [ ]:
def cari_jawaban(pertanyaan_pengguna):    """Mencari jawaban yang maknanya paling dekat dengan pertanyaan pengguna."""    vektor_pengguna = ubah_ke_vektor(pertanyaan_pengguna)    skor = kemiripan(vektor_pengguna, vektor_pengetahuan)[0]    posisi_terbaik = int(skor.argmax())    return {        "jawaban": daftar_jawaban[posisi_terbaik],        "pertanyaan_termirip": daftar_pertanyaan[posisi_terbaik],        "skor": float(skor[posisi_terbaik]),    }# Uji coba pertamacontoh = "kapan borobudur dibangun" if TEMA == "wisata" else "kapan perang dunia kedua mulai"hasil = cari_jawaban(contoh)print("Pertanyaan  :", contoh)print("Jawaban     :", hasil["jawaban"])print()print("Cocok dengan:", hasil["pertanyaan_termirip"])print("Skor        :", round(hasil["skor"], 3))

Perhatikan bahwa pertanyaan uji tadi ditulis tanpa huruf kapital dan tanpa tanda tanya,susunannya juga berbeda dari yang ada di tabel. Chatbot tetap menemukan jawaban yang benar.

---## TUGAS 1. Uji dengan kalimatmu sendiriTulis lima pertanyaan versi kamu sendiri di dalam tanda kutip pada sel di bawah.Syaratnya: susunan katanya harus berbeda dari pertanyaan di tabel, tapi maksudnya sama.Contoh kalau temanya wisata:- tabel berisi `"Berapa tinggi Menara Eiffel?"`- kamu tulis `"eiffel itu tingginya berapa ya"`Setelah itu jalankan selnya dan periksa: apakah semua jawabannya tepat?

In [ ]:
# TUGAS 1 - versi kuncicontoh_uji = {    "wisata": [        "eiffel itu tingginya berapa ya",        "borobudur letaknya dimana sih",        "siapa sih yang bikin taj mahal",        "hewan komodo bisa dilihat dimana",        "kenapa raja ampat terkenal banget",    ],    "sejarah": [        "perang dunia pertama itu tahun berapa",        "siapa orang pertama yang jalan di bulan",        "kenapa tembok berlin dirobohkan",        "indonesia merdeka tanggal berapa",        "gandhi itu terkenal karena apa",    ],}pertanyaan_uji = contoh_uji[TEMA]for pertanyaan in pertanyaan_uji:    hasil = cari_jawaban(pertanyaan)    print("Kamu    :", pertanyaan)    print("Chatbot :", hasil["jawaban"])    print("          (skor kemiripan:", round(hasil["skor"], 3), ")")    print("-" * 72)

---## TUGAS 2. Tambah pengetahuan chatbotmu sendiriChatbot hanya tahu apa yang ada di tabelnya. Sekarang tambahkan lima pengetahuan barusesuai tema pilihanmu. Cari faktanya dari sumber yang kamu percaya, jangan mengarang.Isi bagian `pertanyaan` dan `jawaban` pada sel di bawah, lalu jalankan.

In [ ]:
# TUGAS 2 - versi kuncitambahan_kunci = {    "wisata": [        {"pertanyaan": "Di negara mana Menara Pisa berada?",         "jawaban": "Menara Pisa berada di kota Pisa, Italia, dan terkenal karena bangunannya miring."},        {"pertanyaan": "Apa itu Kota Terlarang?",         "jawaban": "Kota Terlarang adalah kompleks istana kaisar Tiongkok di Beijing yang dipakai pada masa Dinasti Ming dan Qing."},        {"pertanyaan": "Di mana letak Pulau Paskah?",         "jawaban": "Pulau Paskah terletak di Samudra Pasifik dan termasuk wilayah Chili. Pulau ini terkenal karena patung batu raksasa bernama moai."},        {"pertanyaan": "Apa yang terkenal dari Kota Venesia?",         "jawaban": "Venesia di Italia terkenal karena dibangun di atas kanal air, sehingga transportasi utamanya memakai perahu."},        {"pertanyaan": "Danau apa yang terdalam di dunia?",         "jawaban": "Danau Baikal di Rusia adalah danau air tawar terdalam di dunia dengan kedalaman lebih dari 1.600 meter."},    ],    "sejarah": [        {"pertanyaan": "Siapa Albert Einstein?",         "jawaban": "Albert Einstein adalah fisikawan kelahiran Jerman yang hidup dari tahun 1879 sampai 1955 dan terkenal karena teori relativitas."},        {"pertanyaan": "Apa itu Perang Diponegoro?",         "jawaban": "Perang Diponegoro atau Perang Jawa berlangsung dari tahun 1825 sampai 1830 antara pasukan Pangeran Diponegoro dan pemerintah kolonial Belanda."},        {"pertanyaan": "Kapan Uni Soviet bubar?",         "jawaban": "Uni Soviet resmi bubar pada Desember 1991 dan pecah menjadi lima belas negara merdeka."},        {"pertanyaan": "Siapa Marie Curie?",         "jawaban": "Marie Curie adalah ilmuwan kelahiran Polandia yang meneliti radioaktivitas dan menjadi orang pertama yang menerima dua Hadiah Nobel di bidang berbeda."},        {"pertanyaan": "Apa itu Sumpah Pemuda?",         "jawaban": "Sumpah Pemuda diikrarkan pada 28 Oktober 1928 dalam Kongres Pemuda Kedua di Batavia dan menegaskan satu tanah air, satu bangsa, dan satu bahasa Indonesia."},    ],}tambahan = tambahan_kunci[TEMA]for butir in tambahan:    if butir["pertanyaan"].strip() == "" or butir["jawaban"].strip() == "":        continue    daftar_pertanyaan.append(butir["pertanyaan"])    daftar_jawaban.append(butir["jawaban"])# Pengetahuan baru harus diubah menjadi vektor jugavektor_pengetahuan = ubah_ke_vektor(daftar_pertanyaan)print("Pengetahuan chatbot sekarang:", len(daftar_pertanyaan), "pasang tanya jawab")print()print("Uji pengetahuan baru:")uji_baru = daftar_pertanyaan[-1]print("Kamu    :", uji_baru)print("Chatbot :", cari_jawaban(uji_baru)["jawaban"])

---## TUGAS 3. Ajari chatbot berkata "tidak tahu"Coba tanyakan sesuatu yang sama sekali di luar tema, misalnya `"berapa harga sepatu futsal"`.Chatbot tetap menjawab, dan jawabannya pasti ngawur. Penyebabnya, chatbot selalu mengambilpertanyaan yang **paling dekat**, walaupun sebenarnya jauh.Perbaikannya memakai **ambang batas**. Kalau skor kemiripan terlalu rendah, chatbotsebaiknya mengaku tidak tahu.Tugasmu: geser nilai `AMBANG` pada sel di bawah sampai menemukan angka yang pas, yaituangka yang menolak pertanyaan di luar tema tapi tetap menjawab pertanyaan yang benar.Catat angka yang menurutmu paling tepat.

In [ ]:
AMBANG = 0.45  # @param {type:"slider", min:0, max:1, step:0.05}def cari_jawaban_dengan_ambang(pertanyaan_pengguna):    """Versi chatbot yang berani mengaku tidak tahu."""    hasil = cari_jawaban(pertanyaan_pengguna)    if hasil["skor"] < AMBANG:        hasil["jawaban"] = "Maaf, aku belum punya pengetahuan tentang itu."    return hasilpertanyaan_luar_tema = [    "berapa harga sepatu futsal",    "siapa nama kucing peliharaan saya",    "resep nasi goreng yang enak",]pertanyaan_dalam_tema = daftar_pertanyaan[:2]print("AMBANG saat ini:", AMBANG)print()print("PERTANYAAN DI LUAR TEMA, seharusnya ditolak")print("-" * 72)for pertanyaan in pertanyaan_luar_tema:    hasil = cari_jawaban_dengan_ambang(pertanyaan)    print("skor", round(hasil["skor"], 3), "|", pertanyaan)    print("     ->", hasil["jawaban"][:70])print()print("PERTANYAAN DALAM TEMA, seharusnya tetap dijawab")print("-" * 72)for pertanyaan in pertanyaan_dalam_tema:    hasil = cari_jawaban_dengan_ambang(pertanyaan)    print("skor", round(hasil["skor"], 3), "|", pertanyaan)    print("     ->", hasil["jawaban"][:70])

---## TUGAS 4 (bonus). Bandingkan dengan cara lamaSebelum ada model AI seperti ini, chatbot dibuat dengan cara mencocokkan kata.Kalau kata yang diketik pengguna tidak ada di daftar, chatbot gagal paham.Sel di bawah membandingkan kedua cara. Ganti isi `pertanyaan_bandingan` dengan pertanyaanyang maksudnya benar tapi memakai kata-kata yang sama sekali berbeda dari tabel,lalu perhatikan mana yang berhasil.

In [ ]:
def cara_lama_cocokkan_kata(pertanyaan_pengguna):    """Chatbot gaya lama: menghitung berapa banyak kata yang sama persis."""    kata_pengguna = set(pertanyaan_pengguna.lower().replace("?", "").split())    skor_tertinggi = -1.0    posisi_terbaik = 0    for posisi, pertanyaan in enumerate(daftar_pertanyaan):        kata_tabel = set(pertanyaan.lower().replace("?", "").split())        jumlah_sama = len(kata_pengguna & kata_tabel)        jumlah_gabungan = len(kata_pengguna | kata_tabel)        skor = jumlah_sama / jumlah_gabungan if jumlah_gabungan else 0.0        if skor > skor_tertinggi:            skor_tertinggi = skor            posisi_terbaik = posisi    return {"jawaban": daftar_jawaban[posisi_terbaik], "skor": skor_tertinggi}pertanyaan_bandingan = {    "wisata": "bangunan tinggi dari besi di paris itu ukurannya seberapa",    "sejarah": "pertikaian besar antarbangsa yang meletus tahun 1914 itu apa namanya",}[TEMA]hasil_lama = cara_lama_cocokkan_kata(pertanyaan_bandingan)hasil_ai = cari_jawaban(pertanyaan_bandingan)print("Pertanyaan:", pertanyaan_bandingan)print()print("CARA LAMA, mencocokkan kata")print("  skor   :", round(hasil_lama["skor"], 3))print("  jawaban:", hasil_lama["jawaban"])print()print("CARA AI, mencocokkan makna")print("  skor   :", round(hasil_ai["skor"], 3))print("  jawaban:", hasil_ai["jawaban"])

---## Bonus. Mengintip cara chatbot memilihChatbot sebenarnya memberi peringkat pada semua pertanyaan di tabel, lalu mengambilyang teratas. Sel di bawah menampilkan tiga kandidat teratas beserta skornya.

In [ ]:
def tiga_kandidat_teratas(pertanyaan_pengguna):    vektor_pengguna = ubah_ke_vektor(pertanyaan_pengguna)    skor = kemiripan(vektor_pengguna, vektor_pengetahuan)[0]    urutan = skor.argsort(descending=True)[:3]    print("Pertanyaan:", pertanyaan_pengguna)    print("-" * 72)    for peringkat, posisi in enumerate(urutan, start=1):        posisi = int(posisi)        print(f"{peringkat}. skor {float(skor[posisi]):.3f} | {daftar_pertanyaan[posisi]}")tiga_kandidat_teratas(daftar_pertanyaan[0])

---## Bagian 4. Ngobrol langsung dengan chatbotmuSekarang coba chatbot buatanmu. Ketik pertanyaan pada kotak yang muncul,lalu tekan Enter. Ketik `selesai` untuk berhenti.

In [ ]:
print("Chatbot siap. Ketik 'selesai' untuk berhenti.")print("=" * 72)while True:    pertanyaan = input("Kamu    : ")    if pertanyaan.lower().strip() in ["selesai", "keluar", "stop", "exit"]:        print("Chatbot : Sampai jumpa!")        break    if pertanyaan.strip() == "":        continue    hasil = cari_jawaban_dengan_ambang(pertanyaan)    print("Chatbot :", hasil["jawaban"])    print()

---## Catatan fasilitator**Perkiraan alokasi waktu praktik**| Bagian | Menit ||---|---|| Bagian 0 sampai 1 (setup dan intuisi vektor) | 8 || Bagian 2 sampai 3 (muat data, bangun chatbot) | 8 || TUGAS 1 | 6 || TUGAS 2 | 10 || TUGAS 3 | 6 || TUGAS 4 dan bonus | sisa waktu |**Pertanyaan pemantik saat diskusi**1. Kenapa skor dua kalimat yang mirip tidak pernah pas 1.000?2. Kalau chatbot ini dipakai sekolah untuk menjawab pertanyaan siswa baru, data apa yang harus disiapkan?3. Apa bedanya chatbot ini dengan ChatGPT? Jawaban yang diharapkan: chatbot ini hanya memilih dari jawaban yang sudah ditulis manusia, sementara ChatGPT menyusun kalimat baru.4. Apa risikonya kalau tabel pengetahuan berisi informasi yang salah?**Angka ambang batas**Nilai 0.45 dipakai sebagai titik awal, bukan hasil pengukuran. Jalankan notebook inisebelum hari pelaksanaan, catat rentang skor yang muncul pada pertanyaan dalam temadan luar tema, lalu sesuaikan angka awalnya kalau perlu.

---## Lampiran A. Kalau data gagal dimuatPakai sel di bawah hanya kalau sel pemuatan data di Bagian 2 gagal.Sel ini meminta kamu mengunggah berkas CSV langsung dari laptopmu.Caranya: buka folder `data` di repo workshop, unduh berkas `faq_wisata.csv` atau`faq_sejarah.csv` ke laptopmu, jalankan sel di bawah, lalu pilih berkas tersebut.

In [ ]:
from google.colab import filesterunggah = files.upload()nama_berkas = list(terunggah.keys())[0]df = pd.read_csv(nama_berkas)print("Berhasil memuat", len(df), "pasang tanya jawab dari", nama_berkas)df.head()

---## Lampiran B. Kalau ingin memakai pustaka sentence-transformersNotebook ini berjalan tanpa pemasangan tambahan. Kalau kamu ingin memakai pustaka`sentence-transformers` yang lazim dipakai di dunia kerja, jalankan sel di bawah,lalu jalankan ulang **sel persiapan** di Bagian 0.Kalau Colab menampilkan tombol **RESTART SESSION** setelah pemasangan, tekan tombol itu,lalu jalankan ulang notebook dari Bagian 0.

In [ ]:
!pip install -q -U sentence-transformersprint("Selesai. Jalankan ulang sel persiapan di Bagian 0.")